# Audit causal feature information and persistent stage errors

Question: what information might the current summaries discard when Deep or REM is repeatedly classified as Light?

Use the original Adam/P3 reference and three completed controlled candidates: patch size two, AdamW, and label smoothing.
Align their saved predictions on the same eligible development targets. Do not retrain or change any model.
Keep the three final-test participants reserved. This audit must not read their sensor observations or prepared feature tables.

Inspect two candidate descriptions without adding them to the training feature schema:

- Heart-rate deviation from the mean of strictly previous eligible epochs
- Mean adjacent axis-vector change within each completed thirty-second interval

The first feature uses at most ten past epochs and requires at least four. Histories reset at participant and segment boundaries.
The second excludes adjacent samples separated by more than 0.2 seconds. It never joins samples across epoch boundaries.
Vector change is not an acceleration derivative or a claim about physiological HRV.

The audit is exploratory development analysis. Apparent separation does not establish a new model improvement or an information ceiling.
Choose one feature family for a later controlled ablation only if the evidence supports a clear hypothesis.
Read [the audit protocol](../docs/causal-feature-audit.md) before enabling the raw-data audit.


## Locate the checkout and declare execution gates

**What:** Find the repository, configure notebook logging, and declare setup and audit switches.

**Why:** Report-only execution must reuse verified cached evidence. The raw-data audit needs explicit opt-in and never trains a model.


In [ ]:
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import JSON, FileLink, display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("causal_feature_audit_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/causal-feature-audit.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()

REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "analysis/causal-feature-audit"  # Select this PR branch before merge
RUN_SETUP = False
RUN_AUDIT = False

logger.info("Repository: %s", ROOT)
logger.info("Setup=%s raw-data audit=%s; no model optimization is available", RUN_SETUP, RUN_AUDIT)


## Optional environment setup

**What:** Optionally clone the working branch and install Astral uv with locked analysis and notebook dependencies.

**Why:** A new machine needs the project modules and plotting tools. The existing local environment can skip this cell.
The audit also needs the prepared development features, completed experiment artifacts, and checksum-verified development motion files.
This environment does not need the optional neural-network dependency group.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group baseline --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")
else:
    logger.info("Environment setup skipped")


## Verify the development-only audit plan

**What:** Read the configured study metadata and inspect participant membership, recipes, and raw-file scope.

**Why:** Scope checks must precede observation access. The plan must exclude the reserved test participants and expose no training action.
The existing eligibility rules, model inputs, saved predictions, and model parameters remain unchanged.


In [ ]:
import pandas as pd

from sleep_classifier.causal_feature_audit import CausalFeatureAudit

experiment = CausalFeatureAudit(ROOT / "configs/causal-feature-audit.yaml")
plan = experiment.plan()
development = set(plan["development_participants"])
reserved_test = set(plan["reserved_test_participants"])
if len(development) != 26 or len(reserved_test) != 3 or development & reserved_test:
    raise ValueError("The audit must use the frozen development cohort and reserve the test set")
if (plan["model_training_performed"] is not False
        or plan["test_observations_read"] is not False
        or plan["candidate_features_integrated"] is not False):
    raise ValueError("The audit must remain read-only with respect to models and final-test data")
display(JSON(plan, expanded=False))
logger.info("Audit plan verified: %d development participants, %d reserved test participants",
            len(development), len(reserved_test))


## Optional reproducible raw-data audit

**What:** Explicitly run the audit CLI once, then retain its manifest, tables, figures, and portable dashboard.

**Why:** Motion summaries require checksum-verified raw development observations. Report refreshes should reuse the immutable cached output.
Keep RUN_AUDIT false after a successful run. Existing output blocks accidental replacement; inspect partial output before recovery.
The command performs feature analysis only. It does not fit a model, select a new checkpoint, or write new prepared training features.


In [ ]:
if RUN_AUDIT:
    %cd "{ROOT}"
    !uv run --locked --group baseline --group notebook python -u \
        -m sleep_classifier.causal_feature_audit \
        --config configs/causal-feature-audit.yaml --run
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Feature audit failed; inspect retained output before retrying")
else:
    logger.info("Raw-data audit skipped; load the verified cached report below")


## Load and verify the cached audit

**What:** Load the cached target table and summary through the audit reader's output-hash checks.

**Why:** A dashboard must describe a consistent snapshot rather than silently combine changed files.
The reader verifies registered output files. The participant check below also guards against an unexpected test-set row.
Report-only execution does not rerun raw-file processing when the cache is absent.


In [ ]:
rows = None
summary = None
cached_paths = [experiment.output_dir / "manifest.json", experiment.output_dir / "summary.json"]
if any(path.exists() for path in cached_paths) and not all(path.exists() for path in cached_paths):
    raise RuntimeError("The cached audit is partial; inspect it before creating another snapshot")
if all(path.exists() for path in cached_paths):
    rows, summary = experiment.load_report()
    observed = set(rows["participant_id"].astype(str))
    if not observed <= development or observed & reserved_test:
        raise ValueError("The cached audit contains a participant outside the development scope")
    overview_keys = (
        "created_utc", "participants", "targets", "scope", "test_observations_read",
        "model_training_performed", "candidate_features_integrated",
    )
    display(JSON({key: summary[key] for key in overview_keys}, expanded=True))
    logger.info("Provenance records %d saved-artifact hashes and %d development motion reads",
                len(summary["source_artifact_hashes"]), len(summary["raw_files_read"]))
    logger.info("Verified cached audit: %d targets from %d development participants",
                len(rows), len(observed))
else:
    logger.info("No cached audit exists; enable RUN_AUDIT only when ready for development analysis")


## Count persistent Deep and REM errors

**What:** Show target-class counts and participant counts for persistent Deep/REM-to-Light errors.

**Why:** Aggregate accuracy can hide which people and minority stages remain difficult across controlled recipe changes.
Persistent means that every included recipe predicts Light for the same true Deep or REM target.
The recipes share data and architecture ancestry. Agreement between them is not independent evidence or proof of an incorrect label.


In [ ]:
if rows is not None:
    display(pd.DataFrame(summary["persistent_counts"]))
    participant_counts = rows.groupby("participant_id", sort=True).agg(
        eligible_epochs=("time", "size"),
        persistent_to_light=("persistent_to_light", "sum"),
        reference_accuracy=("reference_correct", "mean"),
    ).sort_values("persistent_to_light", ascending=False)
    display(participant_counts)
else:
    logger.info("Persistent-error tables are pending the first audit snapshot")


## Inspect candidate availability and training-fold summaries

**What:** Inspect candidate feature availability, class summaries fitted on training-fold participants, and matched error groups.

**Why:** A feature cannot silently remove targets when history or sensor pairs are insufficient.
Unavailable heart-rate history and motion changes stay missing. Observed zero movement remains a distinct numeric zero.
Training-fold statistics use training participants only. Error-group comparisons remain exploratory development evidence.
Class separation in these tables does not prove a useful classifier or justify selecting a feature from one participant alone.


In [ ]:
if rows is not None:
    display(pd.DataFrame(summary["candidate_availability"]))
    display(pd.DataFrame(summary["training_fold_stage_summary"]))
    display(pd.DataFrame(summary["matched_error_summary"]))
else:
    logger.info("Availability and descriptive feature tables are pending the first audit snapshot")


## Visualize feature distributions and persistent errors

**What:** Plot the cached descriptive distributions and persistent-error counts through the reusable visualization module.

**Why:** These figures expose distribution overlap, supports, and participant variation before any candidate enters model training.
The plots use existing observations and saved predictions. They do not fit a model or alter the cached snapshot.


In [ ]:
import matplotlib.pyplot as plt

from sleep_classifier.audit_visualization import AuditVisualization

if rows is not None:
    visualization = AuditVisualization(rows, summary)
    for figure in (visualization.plot_distributions(), visualization.plot_persistent_errors()):
        display(figure)
        plt.close(figure)
else:
    logger.info("Audit figures are pending the first verified snapshot")


## Open the portable local dashboard

**What:** Link the dashboard written and hashed by the audit run.

**Why:** The standalone viewer connects participant timelines, recipe predictions, feature values, and persistent-error inspection.
Use its participant, recipe, and window controls to inspect the same snapshot.
Error filtering narrows the table; it does not compress the timeline or remove time gaps.
The viewer runs locally without an external service. This notebook must not rewrite a hashed dashboard after loading the report.


In [ ]:
if rows is not None:
    dashboard = (experiment.output_dir / "dashboard.html").resolve()
    if not dashboard.is_file():
        raise FileNotFoundError("The verified audit snapshot lacks its dashboard")
    display(FileLink(str(dashboard), result_html_prefix="Portable dashboard: "))
else:
    logger.info("The dashboard link is pending the first audit snapshot")


## Record the decision boundary

**What:** Summarize what this audit can establish before proposing one feature ablation.

**Why:** Adaptive development analysis must not be presented as independent final-test evidence or an immediate model improvement.
Inspect persistent errors, candidate support, participant variation, and the train-only descriptive summaries together.
Choose one feature family only after stating a causal, testable hypothesis and its handling of missing history or sample gaps.
Keep the original Adam/P3 recipe fixed for that later comparison. Keep all final-test observations untouched.
No new feature is integrated into the training feature schema by this notebook.


In [ ]:
if summary is not None:
    logger.info("Audit complete: descriptive feature evidence only; no model score gain is claimed")
    logger.info("Next decision: select one feature family only if the audited evidence supports it")
else:
    logger.info("Decision pending: run and inspect the development-only audit first")


## Export the executed notebook as HTML

**What:** Execute the source notebook from a fresh kernel and export its verified cached report as standalone HTML.

**Why:** Keep an inspectable report while source notebooks remain output-free in Git.
Automatic export skips setup, the raw-data audit, and this recursive export cell.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group baseline --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/28_causal_feature_audit.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Causal feature audit notebook export failed")
